# Rule A/B Ablation

Compares four methods on the same **21 tuning images**:
- K-Means only
- K-Means + Rule A
- K-Means + Rule B
- K-Means + Rule A + B

The notebook reports IoU, Dice and blemish-percentage error only. It does **not decide which rule to keep**.


In [1]:
from pathlib import Path
from dataclasses import replace
from itertools import product
import random
import sys

import cv2
import numpy as np
import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in [start, *start.parents]:
        if (
            (candidate / "banana_quality" / "config.py").exists()
            and (candidate / "data" / "annotations").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not find project root. Start Jupyter from inside the banana project."
    )

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from banana_quality.config import ProcessingConfig
from banana_quality.segmentation.kmeans import prepare_image
from banana_quality.segmentation.mask_refinement import banana_mask_from_prepared
from banana_quality.blemish.detection import blemish_mask_from_prepared
from banana_quality.surface_quality import blemish_percentage
from banana_quality.morphology import _morphology, _remove_tiny_components

DATASET_ROOT = PROJECT_ROOT / "data" / "annotations"
RESULTS_DIR = PROJECT_ROOT / "evaluation" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SPLIT_SEED = 42
FINAL_EVALUATION_COUNT = 9

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)


Project root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system
Dataset root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system\data\annotations


In [2]:
images_dir = DATASET_ROOT / "images"
image_paths = sorted(
    p for p in images_dir.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

shuffled_paths = image_paths.copy()
rng = random.Random(SPLIT_SEED)
rng.shuffle(shuffled_paths)
final_names = {p.name for p in shuffled_paths[:FINAL_EVALUATION_COUNT]}

split_df = pd.DataFrame([
    {
        "image": p.name,
        "split": "final_evaluation" if p.name in final_names else "tuning",
    }
    for p in image_paths
]).sort_values(["split", "image"]).reset_index(drop=True)

split_df.to_csv(RESULTS_DIR / "dataset_split.csv", index=False)

assert len(image_paths) == 30, f"Expected 30 images, found {len(image_paths)}"
assert int((split_df["split"] == "tuning").sum()) == 21
assert int((split_df["split"] == "final_evaluation").sum()) == 9

print(split_df["split"].value_counts())


split
tuning              21
final_evaluation     9
Name: count, dtype: int64


In [3]:
def matching_mask(folder: Path, image_path: Path) -> Path:
    path = folder / f"{image_path.stem}.png"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def load_binary_mask(path: Path, target_shape: tuple[int, int]) -> np.ndarray:
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Cannot read mask: {path}")
    mask = np.where(mask > 0, 255, 0).astype(np.uint8)
    if mask.shape != target_shape:
        h, w = target_shape
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    return mask


def resize_binary_mask(mask: np.ndarray, target_shape: tuple[int, int]) -> np.ndarray:
    h, w = target_shape
    resized = cv2.resize(
        np.asarray(mask, dtype=np.uint8),
        (w, h),
        interpolation=cv2.INTER_NEAREST,
    )
    return np.where(resized > 0, 255, 0).astype(np.uint8)


def iou_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    union = int(np.logical_or(pred, gt).sum())
    if union == 0:
        return 1.0
    return float(np.logical_and(pred, gt).sum() / union)


def dice_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    denominator = int(pred.sum() + gt.sum())
    if denominator == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, gt).sum() / denominator)


def load_samples(split_name: str):
    names = set(split_df.loc[split_df["split"] == split_name, "image"])
    samples = []
    for image_path in image_paths:
        if image_path.name not in names:
            continue
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Cannot read image: {image_path}")
        shape = image.shape[:2]
        banana_gt = load_binary_mask(
            matching_mask(DATASET_ROOT / "banana_masks", image_path),
            shape,
        )
        blemish_gt = load_binary_mask(
            matching_mask(DATASET_ROOT / "blemish_masks", image_path),
            shape,
        )
        samples.append({
            "name": image_path.name,
            "image": image,
            "shape": shape,
            "banana_gt": banana_gt,
            "blemish_gt": blemish_gt,
        })
    return samples


def clean_variant(raw_mask, banana_mask, opening, closing, min_area_ratio):
    raw = np.where(np.asarray(raw_mask) > 0, 255, 0).astype(np.uint8)
    opened = _morphology(raw, cv2.MORPH_OPEN, int(opening))
    closed = _morphology(opened, cv2.MORPH_CLOSE, int(closing))
    minimum_area = max(
        8,
        int((np.asarray(banana_mask) > 0).sum() * float(min_area_ratio)),
    )
    final_mask = _remove_tiny_components(closed, minimum_area)
    final_mask[np.asarray(banana_mask) == 0] = 0
    return final_mask


def variant_mask_from_steps(
    steps,
    banana_mask,
    use_rule_a,
    use_rule_b,
    opening,
    closing,
    min_area_ratio,
):
    candidate = np.asarray(steps["kmeans_region_candidate"]) > 0
    if use_rule_a:
        candidate |= np.asarray(steps["rule_a_mask"]) > 0
    if use_rule_b:
        candidate |= np.asarray(steps["rule_b_mask"]) > 0
    return clean_variant(
        np.where(candidate, 255, 0).astype(np.uint8),
        banana_mask,
        opening,
        closing,
        min_area_ratio,
    )


## 1. Load the tuned core parameters from Notebook 01

The core values are selected by parameter tuning. This notebook does **not** retune them.


In [4]:
core_path = RESULTS_DIR / "recommended_core_parameters.csv"
if not core_path.exists():
    raise FileNotFoundError(
        "Run 01_parameter_tuning.ipynb first. Missing: " + str(core_path)
    )

core = pd.read_csv(core_path).iloc[0]

BANANA_K = int(core["kmeans_k"])
BANANA_ACCEPTANCE = float(core["banana_cluster_score_threshold"])
BANANA_OPENING = int(core["banana_opening_kernel"])
BANANA_CLOSING = int(core["banana_closing_kernel"])
BLEMISH_K = int(core["blemish_kmeans_k"])
BLEMISH_CHROMA = float(core["blemish_cluster_chroma_threshold"])
BLEMISH_OPENING = int(core["blemish_opening_kernel"])
BLEMISH_CLOSING = int(core["blemish_closing_kernel"])

print("Loaded tuned core parameters from Notebook 01:")
display(pd.DataFrame([{
    "banana_k": BANANA_K,
    "banana_acceptance": BANANA_ACCEPTANCE,
    "banana_opening": BANANA_OPENING,
    "banana_closing": BANANA_CLOSING,
    "blemish_k": BLEMISH_K,
    "blemish_chroma": BLEMISH_CHROMA,
    "blemish_opening": BLEMISH_OPENING,
    "blemish_closing": BLEMISH_CLOSING,
}]))

DEFAULTS = ProcessingConfig()
core_config = replace(
    DEFAULTS,
    kmeans_k=int(BANANA_K),
    banana_cluster_score_threshold=float(BANANA_ACCEPTANCE),
    blemish_kmeans_k=int(BLEMISH_K),
    blemish_cluster_chroma_threshold=float(BLEMISH_CHROMA),
)


Loaded tuned core parameters from Notebook 01:


,banana_k,banana_acceptance,banana_opening,banana_closing,blemish_k,blemish_chroma,blemish_opening,blemish_closing
0,7,0.35,3,7,6,21.0,3,3


## 2. Prepare the 21 tuning images


In [5]:
tuning_samples = load_samples("tuning")
cache = []
for sample in tuning_samples:
    prepared = prepare_image(sample["image"], core_config)
    banana_steps = banana_mask_from_prepared(
        prepared,
        opening_kernel_size=int(BANANA_OPENING),
        closing_kernel_size=int(BANANA_CLOSING),
    )
    banana_working = np.asarray(banana_steps["banana_mask"], dtype=np.uint8)
    banana_pred = resize_binary_mask(banana_working, sample["shape"])
    cache.append({**sample, "prepared": prepared, "banana_working": banana_working, "banana_pred": banana_pred})
print("Prepared:", len(cache))


Prepared: 21


## 3. Run the four variants


In [6]:
variants = [
    ("K-Means only", False, False),
    ("K-Means + Rule A", True, False),
    ("K-Means + Rule B", False, True),
    ("K-Means + Rule A + B", True, True),
]
rows = []
for sample in cache:
    steps = blemish_mask_from_prepared(
        sample["prepared"],
        sample["banana_working"],
        opening_kernel_size=int(BLEMISH_OPENING),
        closing_kernel_size=int(BLEMISH_CLOSING),
        config=core_config,
    )
    gt = sample["blemish_gt"]
    gt_pct = blemish_percentage(sample["banana_gt"], gt)
    for method, use_a, use_b in variants:
        working = variant_mask_from_steps(
            steps,
            sample["banana_working"],
            use_rule_a=use_a,
            use_rule_b=use_b,
            opening=BLEMISH_OPENING,
            closing=BLEMISH_CLOSING,
            min_area_ratio=core_config.min_blemish_area_ratio,
        )
        pred = resize_binary_mask(working, sample["shape"])
        pred_pct = blemish_percentage(sample["banana_pred"], pred)
        rows.append({
            "image": sample["name"],
            "method": method,
            "iou": iou_score(pred, gt),
            "dice": dice_score(pred, gt),
            "gt_blemish_pct": gt_pct,
            "predicted_blemish_pct": pred_pct,
            "blemish_pct_error": abs(pred_pct - gt_pct),
        })

ablation_per_image = pd.DataFrame(rows)
ablation_summary = (
    ablation_per_image.groupby("method", as_index=False)
    .agg(
        mean_iou=("iou", "mean"),
        mean_dice=("dice", "mean"),
        blemish_percentage_mae=("blemish_pct_error", "mean"),
    )
)

ablation_per_image.to_csv(RESULTS_DIR / "rule_ablation_per_image.csv", index=False)
ablation_summary.to_csv(RESULTS_DIR / "rule_ablation_summary.csv", index=False)
display(ablation_summary)
print("Saved rule_ablation_per_image.csv and rule_ablation_summary.csv")


,method,mean_iou,mean_dice,blemish_percentage_mae
0,K-Means + Rule A,0.363471,0.469886,7.633612
1,K-Means + Rule A + B,0.386514,0.496670,7.615392
2,K-Means + Rule B,0.375197,0.479873,7.758091
3,K-Means only,0.322921,0.416506,7.796957


Saved rule_ablation_per_image.csv and rule_ablation_summary.csv


## End of Notebook 02

Use the ablation table to decide whether Rule A, Rule B, both, or neither should be retained. The notebook reports the measurements but leaves the methodological decision to you.
